# PRE-DATA PROCESSING
DRUG DATASET: ONCOLOGICS

In [1]:
import pandas
import os

In [2]:
# DIRECTORIES
notebook_dir = os.getcwd()
base_dir = os.path.dirname(notebook_dir) # the notebook is in a subdirectory of the base directory
input_dir = os.path.join(base_dir, 'data', 'input', 'oncologics')
print(f"Base directory: {base_dir}")
print(f"Input directory: {input_dir}")

Base directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\DREXPA
Input directory: c:\Users\viviamsb\OneDrive - NTNU\PhD Folder\Pipeline\DREXPA\data\input\oncologics


In [3]:
# Load the oneilson dataset from csv file
drugdata_file = input_dir + '/lab_screen_hsa.csv'
drugdata_df = pandas.read_csv(drugdata_file)
drugdata_df.columns

Index(['drug_name_A', 'drug_name_B', 'targets_A', 'targets_B', 'CAR1', 'LS123',
       'T84', 'NCI-H508', 'NCI-H716', 'NCI-H747', 'SNU-81', 'SW620', 'SW837',
       'SW948', 'SW1417', 'SW1463', 'C2BBe1', 'CL-11', 'HT-115', 'SW1116'],
      dtype='str')

In [4]:
# UNIQUE DRUG COMBINATIONS IN DATASET
drug_combinations = drugdata_df[['drug_name_A', 'drug_name_B']].copy()
drug_combinations = drug_combinations.drop_duplicates()
drug_combinations = drug_combinations.reset_index(drop=True)
print(f'Total unique drug combinations in dataset: {len(drug_combinations)}')

Total unique drug combinations in dataset: 44


In [5]:
# UNIQUE DRUGS IN DATASET
drugs_names = drugdata_df[['drug_name_A', 'drug_name_B']].copy()
drugs_names['drug_name_A'] = drugs_names['drug_name_A'].str.upper()
drugs_names['drug_name_B'] = drugs_names['drug_name_B'].str.upper()

# Make list of total drugs in oneils dataset
drug_list = sorted(set(drugs_names['drug_name_A'].tolist() + drugs_names['drug_name_B'].tolist()))
print(f'Total unique drugs in dataset: {len(drug_list)}')

# Make drug and targets dataframe
drugA_targets_df = drugdata_df[['drug_name_A', 'targets_A']].copy()
drugB_targets_df = drugdata_df[['drug_name_B', 'targets_B']].copy()
drugA_targets_df = drugA_targets_df.rename(columns={'drug_name_A': 'drug_name', 'targets_A': 'targets'})
drugB_targets_df = drugB_targets_df.rename(columns={'drug_name_B': 'drug_name', 'targets_B': 'targets'})
drug_targets_df = pandas.concat([drugA_targets_df, drugB_targets_df], ignore_index=True)
drug_targets_df = drug_targets_df.drop_duplicates()
drug_targets_df = drug_targets_df.reset_index(drop=True)

drugtargets_file = input_dir + '/oncologics_drug_targets.csv'
# drug_targets_df.to_csv(drugtargets_file, index=False, header=True)

# Save list as dataframe of single column 'drug_name'
drug_df = pandas.DataFrame(drug_list, columns=['drug_name'])
drug_file = input_dir + '/oncologics_drug_names.txt'
# drug_df.to_csv(drug_file, index=False, header=True)

Total unique drugs in dataset: 19


In [6]:
# UNIQUE CELL LINES IN DATASET

# In this dataset format, the cell line names are in the column names, so we need to extract them from the column names
cell_names = drugdata_df.columns[4:] # the first 4 columns are drug_name_A, drug_name_B, target_A, target_B
celllines = cell_names.tolist()
celllines.sort()
print(f'Total unique cell lines in dataset: {len(celllines)}')

Total unique cell lines in dataset: 16


In [7]:
# MELT THE DATASET TO LONG FORMAT
drugdata_long_df = drugdata_df.melt(id_vars=['drug_name_A', 'drug_name_B', 'targets_A', 'targets_B'], var_name='cell_line', value_name='synergy')
print(f"Long format dataset shape: {drugdata_long_df.shape}")

# Clean cell line names by removing any '-' and set upper case
drugdata_long_df['cell_line'] = drugdata_long_df['cell_line'].str.replace('-', '').str.upper()

drugdata_nodoses_file = input_dir + '/synergy_data_nodoses.csv'
# drugdata_long_df.to_csv(drugdata_nodoses_file, index=False)

Long format dataset shape: (704, 6)


In [8]:
# PROCESS DRUG DOSES
drugA_doses_file = input_dir + '/drugA_doses_batch1.csv'
drugB_doses_file = input_dir + '/drugB_doses_batch1.csv'
drugA_doses_df = pandas.read_csv(drugA_doses_file)
drugB_doses_df = pandas.read_csv(drugB_doses_file)

# Take the hightest dose for each drug (Conc1)
drugA_doses_df = drugA_doses_df[['drugA', 'A_Conc1']].copy()
drugB_doses_df = drugB_doses_df[['drugB', 'B_Conc1']].copy()

# Remove duplicates and reset index
drugA_doses_df = drugA_doses_df.drop_duplicates().reset_index(drop=True)
drugB_doses_df = drugB_doses_df.drop_duplicates().reset_index(drop=True)

# Concatenate the two dataframes and rename columns to 'drug_name' and 'dose'
drugA_doses_df = drugA_doses_df.rename(columns={'drugA': 'drug_name', 'A_Conc1': 'dose'})
drugB_doses_df = drugB_doses_df.rename(columns={'drugB': 'drug_name', 'B_Conc1': 'dose'})
drug_doses_df = pandas.concat([drugA_doses_df, drugB_doses_df], ignore_index=True)

# Create a dictionary of drug doses for easy lookup
drug_dose_dict = dict(zip(drug_doses_df['drug_name'], drug_doses_df['dose']))

# drug_doses_df.to_csv(input_dir + '/oncologics_drug_doses.csv', index=False)
drug_doses_df

,drug_name,dose
0,Ipatasertib,10000
1,MK-2206,10000
2,PF-477736,625
3,Prexasertib,59
4,AZD-8055,1250
5,Everolimus,9375
6,Torin 1,205
7,Lapatinib,5000
8,Neratinib,3750
9,Niraparib,10000


In [9]:
# ADD DRUG DOSES TO LONG FORMAT DATASET using the drug_dose_dict
drugdata_long_df['conc_A'] = drugdata_long_df['drug_name_A'].map(drug_dose_dict)
drugdata_long_df['conc_B'] = drugdata_long_df['drug_name_B'].map(drug_dose_dict)

# drugdata_long_df.to_csv(input_dir + '/synergy_data_bliss.csv', index=False)
drugdata_long_df

,drug_name_A,drug_name_B,targets_A,targets_B,cell_line,synergy,conc_A,conc_B
0,AZD-8055,Capivasertib,"RPTOR, PRR5",AKT1,CAR1,5.841557,1250,10000
1,AZD-8055,Ipatasertib,"RPTOR, PRR5",AKT1,CAR1,11.209185,1250,10000
2,AZD-8055,Lapatinib,"RPTOR, PRR5","ERBB2, EGFR",CAR1,18.304460,1250,5000
3,AZD-8055,Navitoclax,"RPTOR, PRR5","BCL2, BCL2L1",CAR1,16.920492,1250,3750
4,AZD-8055,Neratinib,"RPTOR, PRR5",ERBB2,CAR1,19.011996,1250,3750
...,...,...,...,...,...,...,...,...
699,Torin 1,Obatoclax,"RPTOR, PRR5",BCL2,SW1116,1.062382,205,625
700,Torin 1,Oxaliplatin,"RPTOR, PRR5",chemo,SW1116,0.811762,205,20000
701,Torin 1,Palbociclib,"RPTOR, PRR5","CCND1, CCND2, CCNK, CCNT1, CDK2, CDK4, CDK6,",SW1116,-6.326221,205,5000
702,Torin 1,SN-38,"RPTOR, PRR5",chemo,SW1116,-5.663663,205,625


In [10]:
## ONCE DREXPA HAS BEEN RUN, ADD DRUG PROFILES
output_dir = os.path.join(base_dir, 'data', 'output', 'oncologics_22panel')
drugprofiles_file = os.path.join(output_dir, 'drug_profiles.csv')

drug_profiles_df = pandas.read_csv(drugprofiles_file)
drugprofiles_dict = dict(zip(drug_profiles_df['drug_name'], drug_profiles_df['PD_profile']))

# Map the drug profiles to the long format dataset
drugdata_long_df['PD_A'] = drugdata_long_df['drug_name_A'].map(drugprofiles_dict)
drugdata_long_df['PD_B'] = drugdata_long_df['drug_name_B'].map(drugprofiles_dict)
drugdata_long_df.to_csv(input_dir + '/synergy_data_22panel.csv', index=False)